Solve the system [[1,2,3],[4,5,6],[7,8,10]] x = [6, 15, 27] using your Gaussian elimination, your LU solver, and np.linalg.solve. Verify all three give the same answer within floating-point tolerance.

In [1]:
import numpy as np

def gaussian_elimination(A, b):
    n = len(b)
    Ab = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])

    for k in range(n):
        max_row = k + np.argmax(np.abs(Ab[k:, k]))
        Ab[[k, max_row]] = Ab[[max_row, k]]

        if abs(Ab[k, k]) < 1e-12:
            raise ValueError(f"Matrix is singular or nearly singular at pivot {k}")

        for i in range(k + 1, n):
            m = Ab[i, k] / Ab[k, k]
            Ab[i, k:] -= m * Ab[k, k:]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]

    return x

In [2]:
def lu_decompose(A):
    n = A.shape[0]
    L = np.eye(n)
    U = A.astype(float).copy()
    P = np.eye(n)

    for k in range(n):
        max_row = k + np.argmax(np.abs(U[k:, k]))
        if max_row != k:
            U[[k, max_row]] = U[[max_row, k]]
            P[[k, max_row]] = P[[max_row, k]]
            if k > 0:
                L[[k, max_row], :k] = L[[max_row, k], :k]

        for i in range(k + 1, n):
            L[i, k] = U[i, k] / U[k, k]
            U[i, k:] -= L[i, k] * U[k, k:]

    return P, L, U

def lu_solve(P, L, U, b):
    n = len(b)
    Pb = P @ b.astype(float)

    y = np.zeros(n)
    for i in range(n):
        y[i] = Pb[i] - L[i, :i] @ y[:i]

    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]

    return x

In [10]:
A=np.array([[1,2,3],
   [4,5,6],
   [7,8,10]],dtype=float)
x=np.array([6,15,27],dtype=float)

In [11]:
a = gaussian_elimination(A, x)
print("Solution:", a)
print("Residual (A@x - b):", A @ a - x)

Solution: [ 3. -3.  3.]
Residual (A@x - b): [0. 0. 0.]


In [14]:
P, L, U = lu_decompose(A)
lu = lu_solve(P, L, U, x)
print("Solution:", lu)
print("Residual (A@x - b):", A @ lu - x)

Solution: [ 3. -3.  3.]
Residual (A@x - b): [0. 0. 0.]


In [13]:
np.linalg.solve(A,x)

array([ 3., -3.,  3.])

Generate a 50x5 random matrix X and target y = X @ w_true + noise. Solve for w using normal equations, QR (via np.linalg.qr), SVD (via np.linalg.svd), and np.linalg.lstsq. Compare all four solutions. Measure the condition number of X^T X and explain how it affects which method you trust.

In [20]:
rng=np.random.default_rng(0)
n,d=50,5
X=rng.standard_normal((n,d))
w_true=rng.standard_normal(d)
noise=0.1 * rng.standard_normal(n)
y=X@w_true + noise

In [25]:
X.shape

(50, 5)

In [23]:
w_true

array([ 0.20191   , -0.03883504,  1.06632455, -0.92163392,  0.80471693])

In [21]:
# 1) Normal equations
w_ne = np.linalg.solve(X.T @ X, X.T @ y)

# 2) QR
Q, R = np.linalg.qr(X)
w_qr = np.linalg.solve(R, Q.T @ y)

# 3) SVD
U, s, Vt = np.linalg.svd(X, full_matrices=False)
w_svd = Vt.T @ ((U.T @ y) / s)

# 4) lstsq
w_lstsq, *_ = np.linalg.lstsq(X, y, rcond=None)

In [22]:
# Compare
sols = {"normal": w_ne, "qr": w_qr, "svd": w_svd, "lstsq": w_lstsq}
for k, w in sols.items():
    print(f"{k:>7}: ||w - w_true|| = {np.linalg.norm(w - w_true):.3e}, "
          f"||Xw - y|| = {np.linalg.norm(X @ w - y):.3e}")

print("\nMax pairwise diff:",
      max(np.linalg.norm(sols[a]-sols[b]) for a in sols for b in sols))

# Condition number
cond_XtX = np.linalg.cond(X.T @ X)
cond_X   = np.linalg.cond(X)
print(f"\ncond(X)    = {cond_X:.3e}")
print(f"cond(X^T X) = {cond_XtX:.3e}  (≈ cond(X)^2)")

 normal: ||w - w_true|| = 3.196e-02, ||Xw - y|| = 7.769e-01
     qr: ||w - w_true|| = 3.196e-02, ||Xw - y|| = 7.769e-01
    svd: ||w - w_true|| = 3.196e-02, ||Xw - y|| = 7.769e-01
  lstsq: ||w - w_true|| = 3.196e-02, ||Xw - y|| = 7.769e-01

Max pairwise diff: 1.7596356360313232e-15

cond(X)    = 1.667e+00
cond(X^T X) = 2.781e+00  (≈ cond(X)^2)
